# Notebook 3: Detección de Objetos (Object Detection)

**Autores:** Javier Arroyo | Julia Cano | Paula Durá  
**Asignatura:** Procesamiento de Imágenes  

---

## Objetivo

A diferencia de la clasificación (Notebook 02), que responde "¿qué categoría es esta imagen?", la **detección de objetos** responde tres preguntas simultáneamente:
- **¿Qué objetos** hay en la imagen?
- **¿Dónde están?** (localización mediante *bounding boxes*)
- **¿Con qué confianza** se detectan?

### Enfoques implementados

| # | Modelo | Enfoque | Ventaja principal |
|---|--------|---------|-------------------|
| 1 | YOLOv8 (preentrenado) | Inferencia directa (COCO, 80 clases) | Sin entrenamiento, multi-objeto |
| 2 | CNN Localizador (from scratch) | Regresión de bbox con pseudo-labels | Control total, modelo ligero |

### Métricas
- **IoU** (Intersection over Union): métrica estándar PASCAL VOC (≥0.5 = correcto)
- **Confianza** de detección
- **Cobertura**: porcentaje de imágenes con al menos una detección

> **Nota:** Este notebook usa el dataset aumentado generado en el Notebook 01.

---
## 3.1 Configuración

Importamos PyTorch (para YOLOv8 vía Ultralytics) y TensorFlow/Keras (para el detector from scratch).

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
import random

from PIL import Image, ImageDraw, ImageFont, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

import torch
import torchvision
from torchvision import transforms

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Usar el dataset (original o aumentado)
DATA_DIR = Path("dataset_augmented")
if not DATA_DIR.exists():
    DATA_DIR = Path("dataset")

classes = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print("Directorio de datos:", DATA_DIR)
print("Clases del dataset:", classes)

# Indexar imágenes
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
rows = []
for cls in classes:
    for p in (DATA_DIR / cls).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            rows.append({"path": str(p), "class": cls})

df = pd.DataFrame(rows)
print(f"Total imágenes: {len(df)}")

---
## 3.2 Modelo Preentrenado: YOLOv8

### ¿Por qué YOLOv8?

**YOLO** (*You Only Look Once*) es una familia de detectores *single-shot* que procesan la imagen completa en una sola pasada, logrando detección en tiempo real. YOLOv8 (Ultralytics, 2023) representa el estado del arte actual:

| Característica | Detalle |
|---|---|
| Dataset de preentrenamiento | COCO (330K imágenes, 80 clases) |
| Arquitectura | CSPDarknet53 + C2f modules + Decoupled Head |
| Variante usada | `yolov8n` (nano: 3.2M parámetros) |
| Detección | Multi-objeto con clasificación y localización simultánea |

### Relevancia para nuestro dataset
YOLOv8 puede detectar automáticamente objetos presentes en nuestras categorías:
- **Animales** → perros, gatos, pájaros, caballos...
- **Ciudad** → coches, personas, semáforos, autobuses...
- **Comida** → platos, tenedores, cuchillos, botellas...
- **Playa** → personas, tablas de surf, sombrillas...
- **Naturaleza** → animales, personas (en paisajes)...

In [ ]:
from ultralytics import YOLO

# Cargar modelo YOLOv8 preentrenado (nano para rapidez, se puede usar 's', 'm', 'l')
model_yolo = YOLO("yolov8n.pt")

# Nombres de las clases COCO
coco_names = model_yolo.names
print(f"Clases COCO disponibles: {len(coco_names)}")
print("Ejemplos:", {k: v for k, v in list(coco_names.items())[:20]})

### 3.2.1 Inferencia sobre ejemplos

Ejecutamos YOLOv8 sobre imágenes representativas de cada categoría para visualizar las detecciones con sus *bounding boxes* y niveles de confianza.

In [ ]:
def run_yolo_on_image(model, img_path, conf_threshold=0.3):
    """Ejecuta YOLO sobre una imagen y devuelve las detecciones."""
    results = model(img_path, conf=conf_threshold, verbose=False)
    result = results[0]
    
    detections = []
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0].cpu().numpy())
            cls_id = int(box.cls[0].cpu().numpy())
            cls_name = coco_names[cls_id]
            detections.append({
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                "confidence": conf,
                "class_id": cls_id,
                "class_name": cls_name
            })
    return detections

def plot_detections(img_path, detections, title="", ax=None):
    """Visualiza las detecciones sobre la imagen."""
    img = Image.open(img_path).convert("RGB")
    
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
    ax.imshow(img)
    
    colors = plt.cm.Set3(np.linspace(0, 1, 12))
    
    for i, det in enumerate(detections):
        color = colors[i % len(colors)]
        rect = patches.Rectangle(
            (det["x1"], det["y1"]),
            det["x2"] - det["x1"],
            det["y2"] - det["y1"],
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        label = f"{det['class_name']} ({det['confidence']:.2f})"
        ax.text(det["x1"], det["y1"] - 5, label,
                fontsize=7, color="white",
                bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8))
    
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return ax

In [ ]:
# Ejecutar YOLO sobre 3 imágenes de cada categoría
N_EXAMPLES = 3

fig, axes = plt.subplots(len(classes), N_EXAMPLES, figsize=(5*N_EXAMPLES, 5*len(classes)))

for i, cls in enumerate(classes):
    cls_df = df[df["class"] == cls]
    sample_paths = cls_df["path"].sample(min(N_EXAMPLES, len(cls_df)), random_state=SEED).tolist()
    
    for j, img_path in enumerate(sample_paths):
        detections = run_yolo_on_image(model_yolo, img_path, conf_threshold=0.25)
        n_det = len(detections)
        det_classes = [d["class_name"] for d in detections]
        
        ax = axes[i, j] if len(classes) > 1 else axes[j]
        plot_detections(img_path, detections, 
                       title=f"{cls} | {n_det} objetos: {', '.join(det_classes[:3])}", ax=ax)

plt.suptitle("Detecciones YOLOv8 por categoría", fontsize=16)
plt.tight_layout()
plt.show()

### 3.2.2 Análisis masivo de detecciones

Ejecutamos YOLOv8 sobre **todo el dataset** para obtener estadísticas globales: qué tipos de objetos COCO predominan en cada categoría de nuestro dataset.

In [ ]:
# Detección masiva sobre todo el dataset
all_detections = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="YOLOv8 inference"):
    try:
        dets = run_yolo_on_image(model_yolo, row["path"], conf_threshold=0.25)
        for d in dets:
            d["image_path"] = row["path"]
            d["dataset_class"] = row["class"]
        all_detections.extend(dets)
    except Exception as e:
        pass  # Saltar imágenes problemáticas

det_df = pd.DataFrame(all_detections)
print(f"Total detecciones: {len(det_df)}")
print(f"Clases COCO detectadas: {det_df['class_name'].nunique()}")

In [ ]:
# Top 10 objetos detectados globalmente
top_global = det_df["class_name"].value_counts().head(15)

plt.figure(figsize=(10, 5))
top_global.plot(kind="bar", color="#3498db")
plt.title("Top 15 objetos detectados (todo el dataset)")
plt.ylabel("Número de detecciones")
plt.xlabel("Clase COCO")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Objetos detectados por categoría del dataset
fig, axes = plt.subplots(1, len(classes), figsize=(5*len(classes), 6))

for i, cls in enumerate(classes):
    cls_dets = det_df[det_df["dataset_class"] == cls]
    if len(cls_dets) > 0:
        top = cls_dets["class_name"].value_counts().head(8)
        top.plot(kind="barh", ax=axes[i], color=plt.cm.Set2(np.arange(len(top))))
        axes[i].set_title(f"{cls}\n({len(cls_dets)} detecciones)", fontsize=11)
        axes[i].set_xlabel("count")
    else:
        axes[i].text(0.5, 0.5, "Sin detecciones", ha="center", va="center")
        axes[i].set_title(cls)

plt.suptitle("Objetos COCO detectados por categoría del dataset", fontsize=14)
plt.tight_layout()
plt.show()

### 3.2.3 Estadísticas de confianza y cobertura

Analizamos la **distribución de confianza** de las detecciones y la **cobertura** (porcentaje de imágenes donde YOLOv8 encuentra al menos un objeto).

In [ ]:
# Estadísticas de confianza por categoría
conf_stats = det_df.groupby("dataset_class")["confidence"].agg(["mean","std","min","max","count"])
display(conf_stats)

# Distribución de confianza por categoría
fig, ax = plt.subplots(figsize=(10, 5))
for cls in classes:
    cls_confs = det_df[det_df["dataset_class"] == cls]["confidence"]
    if len(cls_confs) > 0:
        ax.hist(cls_confs, bins=20, alpha=0.4, label=cls)
ax.set_title("Distribución de confianza de detecciones por categoría")
ax.set_xlabel("Confidence")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

# Cobertura: % de imágenes con al menos 1 detección
coverage = {}
for cls in classes:
    cls_paths = set(df[df["class"] == cls]["path"])
    detected_paths = set(det_df[det_df["dataset_class"] == cls]["image_path"].unique())
    coverage[cls] = len(detected_paths) / len(cls_paths) * 100

coverage_df = pd.DataFrame.from_dict(coverage, orient="index", columns=["% imágenes con detección"])
display(coverage_df)

### 3.2.4 Análisis de tamaño de *bounding boxes*

El tamaño de los objetos detectados puede revelar patrones interesantes: en imágenes de animales, el sujeto suele ser grande y centrado, mientras que en ciudad pueden haber muchos objetos pequeños distribuidos.

In [ ]:
# Calcular área relativa de bounding boxes
det_df["box_width"] = det_df["x2"] - det_df["x1"]
det_df["box_height"] = det_df["y2"] - det_df["y1"]
det_df["box_area"] = det_df["box_width"] * det_df["box_height"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de tamaño de bbox
for cls in classes:
    cls_areas = det_df[det_df["dataset_class"] == cls]["box_area"]
    if len(cls_areas) > 0:
        axes[0].hist(cls_areas, bins=30, alpha=0.4, label=cls)
axes[0].set_title("Distribución de área de bounding box")
axes[0].set_xlabel("Área (px²)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Media de detecciones por imagen y categoría
dets_per_img = det_df.groupby(["dataset_class", "image_path"]).size().reset_index(name="n_detections")
mean_dets = dets_per_img.groupby("dataset_class")["n_detections"].mean()
mean_dets.plot(kind="bar", ax=axes[1], color=["#2ecc71","#3498db","#e74c3c","#f39c12","#9b59b6"])
axes[1].set_title("Media de objetos detectados por imagen")
axes[1].set_ylabel("Promedio detecciones")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 3.3 Detector from scratch (CNN Localizador)

### Enfoque: *Teacher-Student* con pseudo-labels

Como **no disponemos de anotaciones manuales** de *bounding boxes*, empleamos una estrategia de transferencia de conocimiento:

1. **Teacher** (YOLOv8): genera *pseudo-anotaciones* detectando el objeto principal de cada imagen
2. **Student** (CNN simple): aprende a replicar esas detecciones con un modelo más ligero

Este patrón es habitual en la industria para crear modelos eficientes a partir de modelos pesados (conocido como *knowledge distillation*).

### Tarea de regresión
El modelo predice 4 coordenadas normalizadas $[x_1, y_1, x_2, y_2] \in [0, 1]$ que definen el *bounding box* del objeto principal.

> **Limitación:** Este detector solo localiza **un objeto por imagen** (el de mayor confianza según YOLOv8).

In [ ]:
# Preparar pseudo-anotaciones: para cada imagen, tomamos la detección principal (mayor confianza)
pseudo_labels = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Generando pseudo-labels"):
    try:
        dets = run_yolo_on_image(model_yolo, row["path"], conf_threshold=0.25)
        if len(dets) > 0:
            # Tomar la detección con mayor confianza
            best = max(dets, key=lambda d: d["confidence"])
            
            # Obtener tamaño de imagen para normalizar
            img = Image.open(row["path"])
            w, h = img.size
            
            pseudo_labels.append({
                "path": row["path"],
                "class": row["class"],
                "x1_norm": best["x1"] / w,
                "y1_norm": best["y1"] / h,
                "x2_norm": best["x2"] / w,
                "y2_norm": best["y2"] / h,
                "det_class": best["class_name"],
                "det_conf": best["confidence"]
            })
    except Exception:
        pass

pseudo_df = pd.DataFrame(pseudo_labels)
print(f"Imágenes con pseudo-anotación: {len(pseudo_df)} / {len(df)}")
print(f"\nDistribución por clase:")
print(pseudo_df["class"].value_counts().sort_index())

In [ ]:
# Split train/val/test para el detector
from sklearn.model_selection import train_test_split

train_det, temp_det = train_test_split(pseudo_df, test_size=0.3, random_state=SEED, stratify=pseudo_df["class"])
val_det, test_det = train_test_split(temp_det, test_size=0.5, random_state=SEED, stratify=temp_det["class"])

print(f"Train: {len(train_det)} | Val: {len(val_det)} | Test: {len(test_det)}")

In [ ]:
# Dataset de TensorFlow para el detector
DET_IMG_SIZE = (128, 128)  # Tamaño más pequeño para el modelo simple
DET_BATCH = 32

def decode_for_det(path, bbox):
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, DET_IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, bbox

def make_det_ds(df_, shuffle=False):
    paths = df_["path"].values
    bboxes = df_[["x1_norm", "y1_norm", "x2_norm", "y2_norm"]].values.astype(np.float32)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, bboxes))
    ds = ds.map(decode_for_det, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(500, seed=SEED)
    ds = ds.batch(DET_BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

train_det_ds = make_det_ds(train_det, shuffle=True)
val_det_ds = make_det_ds(val_det, shuffle=False)
test_det_ds = make_det_ds(test_det, shuffle=False)

print("Datasets creados")

### 3.3.1 Arquitectura del detector

CNN con 4 bloques convolucionales seguidos de capas densas. La salida usa activación **sigmoid** para producir coordenadas en $[0, 1]$.

```
Input(128×128×3) → [Conv32→Pool→BN] → [Conv64→Pool→BN] → [Conv128→Pool→BN] → [Conv256→Pool] → GAP → Dense256 → Dense128 → Sigmoid(4)
```

**Loss:** MSE (Mean Squared Error) — apropiado para regresión de coordenadas continuas.

In [ ]:
# Modelo de detección simple (CNN -> regresión de bbox)
det_inputs = keras.Input(shape=(*DET_IMG_SIZE, 3))

x = layers.Conv2D(32, 3, padding="same", activation="relu")(det_inputs)
x = layers.MaxPool2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPool2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = layers.MaxPool2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
x = layers.MaxPool2D()(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)

# Salida: 4 coordenadas normalizadas [x1, y1, x2, y2] con sigmoid [0,1]
det_outputs = layers.Dense(4, activation="sigmoid")(x)

det_model = keras.Model(det_inputs, det_outputs, name="simple_detector")

det_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

det_model.summary()

In [ ]:
# Entrenar el detector
det_callbacks = [
    keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6)
]

history_det = det_model.fit(
    train_det_ds,
    validation_data=val_det_ds,
    epochs=50,
    callbacks=det_callbacks
)

### 3.3.2 Curvas de entrenamiento

Monitorizamos MSE y MAE (Mean Absolute Error) durante el entrenamiento para verificar la convergencia y detectar *overfitting*.

In [ ]:
# Curvas
h = history_det.history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(h["loss"], label="train_loss (MSE)")
axes[0].plot(h["val_loss"], label="val_loss (MSE)")
axes[0].set_title("Detector from scratch - Loss (MSE)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()

axes[1].plot(h["mae"], label="train_MAE")
axes[1].plot(h["val_mae"], label="val_MAE")
axes[1].set_title("Detector from scratch - MAE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE")
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3.3 Evaluación con IoU (Intersection over Union)

El **IoU** mide el solapamiento entre la predicción y el *ground truth*:

$$\text{IoU} = \frac{\text{Área de Intersección}}{\text{Área de Unión}}$$

Umbrales estándar:
- **IoU ≥ 0.5** → Detección correcta (criterio PASCAL VOC / AP@50)
- **IoU ≥ 0.75** → Detección precisa (criterio estricto / AP@75)

In [ ]:
def compute_iou(box1, box2):
    """Calcula IoU entre dos bboxes [x1,y1,x2,y2] normalizados."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / (union + 1e-8)

# Evaluar en test
ious = []
predictions = []

for batch_imgs, batch_bboxes in test_det_ds:
    pred_bboxes = det_model.predict(batch_imgs, verbose=0)
    
    for gt, pred in zip(batch_bboxes.numpy(), pred_bboxes):
        iou = compute_iou(gt, pred)
        ious.append(iou)
        predictions.append({"gt": gt, "pred": pred, "iou": iou})

ious = np.array(ious)

print(f"IoU medio: {ious.mean():.4f} ± {ious.std():.4f}")
print(f"IoU mediana: {np.median(ious):.4f}")
print(f"% IoU ≥ 0.5 (PASCAL VOC): {(ious >= 0.5).mean()*100:.1f}%")
print(f"% IoU ≥ 0.75 (estricto): {(ious >= 0.75).mean()*100:.1f}%")

In [ ]:
# Distribución de IoU
plt.figure(figsize=(10, 5))
plt.hist(ious, bins=30, color="#3498db", alpha=0.7, edgecolor="black")
plt.axvline(0.5, color="red", linestyle="--", linewidth=2, label="IoU = 0.5 (PASCAL VOC)")
plt.axvline(0.75, color="orange", linestyle="--", linewidth=2, label="IoU = 0.75 (estricto)")
plt.axvline(ious.mean(), color="green", linestyle="-", linewidth=2, label=f"Media = {ious.mean():.3f}")
plt.title("Distribución de IoU del detector from scratch")
plt.xlabel("IoU")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

### 3.3.4 Visualización: predicciones vs *ground truth*

Comparamos visualmente las predicciones del detector (rojo, discontinuo) con las pseudo-labels de YOLOv8 (verde, continuo). El IoU indica la calidad de cada predicción individual.

In [ ]:
# Visualizar predicciones vs ground truth
test_paths = test_det["path"].values
test_gt = test_det[["x1_norm", "y1_norm", "x2_norm", "y2_norm"]].values

# Seleccionar 10 imágenes aleatorias
n_show = 10
indices = random.sample(range(len(test_paths)), min(n_show, len(test_paths)))

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for ax_idx, idx in enumerate(indices):
    img_path = test_paths[idx]
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    
    # Ground truth (pseudo-label de YOLO)
    gt = test_gt[idx]
    
    # Predicción del modelo
    img_resized = img.resize(DET_IMG_SIZE)
    img_arr = np.array(img_resized).astype(np.float32) / 255.0
    pred = det_model.predict(img_arr[np.newaxis], verbose=0)[0]
    
    # IoU
    iou = compute_iou(gt, pred)
    
    axes[ax_idx].imshow(img)
    
    # GT box (verde)
    gt_rect = patches.Rectangle(
        (gt[0]*w, gt[1]*h), (gt[2]-gt[0])*w, (gt[3]-gt[1])*h,
        linewidth=2, edgecolor="lime", facecolor="none", linestyle="-", label="GT (YOLO)"
    )
    axes[ax_idx].add_patch(gt_rect)
    
    # Pred box (rojo)
    pred_rect = patches.Rectangle(
        (pred[0]*w, pred[1]*h), (pred[2]-pred[0])*w, (pred[3]-pred[1])*h,
        linewidth=2, edgecolor="red", facecolor="none", linestyle="--", label="Pred"
    )
    axes[ax_idx].add_patch(pred_rect)
    
    cat = test_det.iloc[idx]["class"] if idx < len(test_det) else "?"
    axes[ax_idx].set_title(f"{cat} | IoU={iou:.2f}", fontsize=9)
    axes[ax_idx].axis("off")

# Leyenda
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='lime', linewidth=2, label='GT (pseudo-label YOLO)'),
    Line2D([0], [0], color='red', linewidth=2, linestyle='--', label='Predicción modelo')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=12)
plt.suptitle("Detector from scratch: Ground Truth vs Predicción", fontsize=14)
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
plt.show()

---
## 3.4 Comparación: Preentrenado vs From Scratch

Comparamos ambos enfoques cuantitativa y cualitativamente para entender las diferencias fundamentales.

In [ ]:
# Comparación de resultados YOLOv8 vs detector simple

# YOLOv8: calcular IoU contra sus propias pseudo-labels (referencia perfecta = 1.0)
# En realidad, para YOLOv8 evaluamos la confianza media y el número de detecciones
yolo_stats = {
    "Modelo": "YOLOv8 (preentrenado)",
    "Tipo": "Preentrenado (COCO, 80 clases)",
    "Detecciones totales": len(det_df),
    "Clases detectadas": det_df["class_name"].nunique(),
    "Confianza media": round(det_df["confidence"].mean(), 4),
    "Multi-objeto": "Sí (múltiples por imagen)",
    "Requiere entrenamiento": "No"
}

scratch_stats = {
    "Modelo": "CNN Detector (from scratch)",
    "Tipo": "Entrenado desde cero",
    "Detecciones totales": len(test_det),
    "Clases detectadas": "1 (objeto principal)",
    "Confianza media": f"IoU={ious.mean():.4f}",
    "Multi-objeto": "No (solo 1 bbox por imagen)",
    "Requiere entrenamiento": "Sí (pseudo-labels)"
}

comparison = pd.DataFrame([yolo_stats, scratch_stats])
display(comparison.T)

In [ ]:
# Gráfico de comparación
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confianza media por categoría (YOLO)
yolo_conf_by_class = det_df.groupby("dataset_class")["confidence"].mean()
yolo_conf_by_class.plot(kind="bar", ax=axes[0], color="#3498db", alpha=0.8)
axes[0].set_title("YOLOv8: Confianza media por categoría")
axes[0].set_ylabel("Confidence")
axes[0].tick_params(axis='x', rotation=45)

# 2. IoU del detector simple por categoría
test_det_with_iou = test_det.copy()
test_det_with_iou["iou"] = ious[:len(test_det)]
iou_by_class = test_det_with_iou.groupby("class")["iou"].mean()
iou_by_class.plot(kind="bar", ax=axes[1], color="#e74c3c", alpha=0.8)
axes[1].set_title("Detector scratch: IoU medio por categoría")
axes[1].set_ylabel("IoU")
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="IoU=0.5")
axes[1].legend()

# 3. Cobertura de detección
coverage_data = pd.DataFrame({
    "YOLOv8": [coverage[cls] for cls in classes],
    "Scratch": [100.0] * len(classes)  # el modelo simple siempre predice
}, index=classes)
coverage_data.plot(kind="bar", ax=axes[2], color=["#3498db", "#e74c3c"], alpha=0.8)
axes[2].set_title("Cobertura: % imágenes con detección")
axes[2].set_ylabel("% cobertura")
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Comparación: YOLOv8 (preentrenado) vs Detector from scratch", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3.5 Conclusiones

### Modelo Preentrenado (YOLOv8)

| Aspecto | Evaluación |
|---------|-----------|
| **Rendimiento** | Excelente — detección multi-objeto con alta confianza |
| **Cobertura** | Alta en categorías con objetos claros (animales, comida) |
| **Ventaja clave** | No requiere entrenamiento ni anotaciones |
| **Limitación** | Solo clases COCO; no detecta conceptos abstractos (escenas) |

### Modelo from scratch (CNN Localizador)

| Aspecto | Evaluación |
|---------|-----------|
| **Rendimiento** | Moderado — aprende patrones básicos de localización |
| **Cobertura** | 100% (siempre produce un bbox) |
| **Ventaja clave** | Control total, modelo ligero y personalizable |
| **Limitación** | Solo 1 bbox por imagen; requiere pseudo-labels |

### Reflexión

La brecha de rendimiento es esperada y educativa:
- **YOLOv8** se beneficia de millones de imágenes anotadas manualmente por expertos
- **El detector simple** solo tiene ~2000 pseudo-labels ruidosas

En un escenario de producción, se haría **fine-tuning de YOLOv8** con anotaciones específicas del dominio, combinando lo mejor de ambos mundos: la potencia del preentrenamiento con la especificidad del dominio.